In [1]:
!pip install streamlit pyngrok xgboost plotly



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 59.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv('/content/pima_indians_diabetes1.csv')

# Features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Models
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(),
    "XGBoost": XGBClassifier()
}

best_model = None
best_accuracy = 0

print("MODEL ACCURACY RESULTS")
print("-" * 40)

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"{name}: {accuracy:.2f}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model

print("\nBest Model Selected Successfully")

MODEL ACCURACY RESULTS
----------------------------------------
Logistic Regression: 0.35
Random Forest: 0.50
SVM: 0.38
KNN: 0.47
XGBoost: 0.45

Best Model Selected Successfully


In [3]:
import joblib

joblib.dump(best_model, 'diabetes_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Model Saved")

Model Saved


In [4]:
%%writefile app.py

import streamlit as st
import joblib
import numpy as np
import plotly.graph_objects as go

# Load model
model = joblib.load('diabetes_model.pkl')
scaler = joblib.load('scaler.pkl')

st.title("DiaLife AI")
st.subheader("Non-Invasive Diabetes Risk Prediction System")

st.write("Enter Patient Details")

# Inputs
preg = st.number_input("Pregnancies", 0, 20)
glucose = st.number_input("Glucose", 0, 300)
bp = st.number_input("Blood Pressure", 0, 200)
skin = st.number_input("Skin Thickness", 0, 100)
insulin = st.number_input("Insulin", 0, 900)
bmi = st.number_input("BMI", 0.0, 70.0)
dpf = st.number_input("Diabetes Pedigree Function", 0.0, 3.0)
age = st.number_input("Age", 1, 120)

if st.button("Predict Diabetes Risk"):

    data = np.array([[preg, glucose, bp, skin,
                      insulin, bmi, dpf, age]])

    data = scaler.transform(data)

    prediction = model.predict(data)[0]

    probability = model.predict_proba(data)[0][1]

    risk = probability * 100

    st.subheader(f"Diabetes Risk Score: {risk:.2f}%")

    # Gauge Chart
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=risk,
        title={'text': "Risk Level"},
        gauge={'axis': {'range': [0, 100]}}
    ))

    st.plotly_chart(fig)

    # Result
    if prediction == 1:
        st.error("High Diabetes Risk Detected")
    else:
        st.success("Low Diabetes Risk")

    # Recommendations
    st.subheader("AI Health Recommendations")

    if bmi > 25:
        st.write("• Reduce weight with exercise")

    if glucose > 140:
        st.write("• Reduce sugar intake")

    if age > 45:
        st.write("• Regular health checkups recommended")

    st.write("• Drink more water")
    st.write("• Walk daily")
    st.write("• Sleep 7-8 hours")

Writing app.py


In [5]:
!streamlit run app.py &>/content/logs.txt &

In [6]:
!pip install pyngrok

In [7]:
from pyngrok import ngrok

ngrok.set_auth_token("3E1RPo7BbTG467rOH3ACCCqD3F8_j3LvKzgmJx7QQU78uC3Q")

In [8]:
!streamlit run app.py &>/content/logs.txt &

In [9]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://mystified-twerp-evacuate.ngrok-free.dev" -> "http://localhost:8501"
